### Alpha Vantage Ingestion

In [0]:
dbutils.widgets.text("tickers", "AAPL,MSFT,GOOGL,AMZN,NVDA", "Tickers (comma separated)")
dbutils.widgets.text("landing_volume_path", "/Volumes/catalog/bronze_landing", "Landing Volume Path")

tickers = [t.strip().upper() for t in dbutils.widgets.get("tickers").split(",") if t.strip()]
landing_path = dbutils.widgets.get("landing_volume_path")

print(f"Ingesting {len(tickers)} tickers: {tickers}")
print(f"Landing path: {landing_path}")

In [0]:
import requests
import time
import json
from datetime import datetime, timezone

# Store the key as a secret: databricks secrets put-secret alpha_vantage api_key
API_KEY = dbutils.secrets.get(scope="alpha_vantage", key="api_key")
BASE_URL = "https://www.alphavantage.co/query"

# 5 requests/min -> stay safely under that with a 13s gap between calls
SECONDS_BETWEEN_CALLS = 13
MAX_RETRIES = 2
RETRY_BACKOFF_SECONDS = 20

In [0]:
def call_alpha_vantage(function, symbol):
    params = {"function": function, "symbol": symbol, "apikey": API_KEY}

    for attempt in range(MAX_RETRIES + 1):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()

            if "Error Message" in payload:
                error_msg = payload["Error Message"]
                print(f"[{symbol}/{function}] invalid request: {error_msg} - skipping, will not retry")
                return None

            if "Note" in payload or "Information" in payload:
                # Rate limit hit - back off and retry
                msg = payload.get("Note") or payload.get("Information")
                print(f"[{symbol}/{function}] rate-limited: {msg}")
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_BACKOFF_SECONDS)
                    continue
                else:
                    print(f"[{symbol}/{function}] giving up after {MAX_RETRIES} retries")
                    return None

            return payload

        except requests.exceptions.RequestException as e:
            print(f"[{symbol}/{function}] request failed: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS)
                continue
            return None

    return None

In [0]:
run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
failures = []
call_count = 0

for symbol in tickers:
    # --- Latest quote ---
    quote_payload = call_alpha_vantage("GLOBAL_QUOTE", symbol)
    call_count += 1
    if quote_payload is not None and quote_payload.get("Global Quote"):
        path = f"{landing_path}/quotes/{symbol}_{run_ts}.json"
        dbutils.fs.put(path, json.dumps(quote_payload), overwrite=True)
        print(f"landed quote -> {path}")
    else:
        failures.append((symbol, "GLOBAL_QUOTE"))

    time.sleep(SECONDS_BETWEEN_CALLS)

    # --- Company overview ---
    overview_payload = call_alpha_vantage("OVERVIEW", symbol)
    call_count += 1
    if overview_payload is not None and overview_payload.get("Symbol"):
        path = f"{landing_path}/company_info/{symbol}_{run_ts}.json"
        dbutils.fs.put(path, json.dumps(overview_payload), overwrite=True)
        print(f"landed company info -> {path}")
    else:
        failures.append((symbol, "OVERVIEW"))

    time.sleep(SECONDS_BETWEEN_CALLS)

print(f"\nDone. {call_count} API calls made. {len(failures)} failures: {failures}")

# Surface failures without crashing the job - a partial run is still useful.
# If every single call failed, that's a real problem (bad key, API down), so raise then.
if failures and len(failures) == call_count:
    raise RuntimeError(f"All {call_count} API calls failed - check API key / Alpha Vantage status.")